In [ ]:
!pip install azure-ai-documentintelligence

In [ ]:
import os
from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from datetime import datetime

ponto_de_extremidade = "SEU_PONTO_DE_EXTREMIDADE_AQUI"
chave = "SUA_CHAVE_AQUI"

url_do_documento = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-invoice.pdf"

In [ ]:
cliente_document_intelligence = DocumentIntelligenceClient(
    endpoint=ponto_de_extremidade, credential=AzureKeyCredential(chave)
)

poller = cliente_document_intelligence.begin_analyze_document_from_url(
    "prebuilt-invoice", url_do_documento
)
resultado = poller.result()

print("Documento analisado com sucesso.")

In [ ]:
relatorio_fraude = []

if resultado.documents:
    documento = resultado.documents[0]
    print("\n--- INFORMAÇÕES EXTRAÍDAS DO DOCUMENTO ---")

    def get_field_value(field_name):
        field = documento.fields.get(field_name)
        if field:
            return field.value if hasattr(field, 'value') else field.content
        return None

    vendor_name = get_field_value("VendorName")
    customer_name = get_field_value("CustomerName")
    invoice_id = get_field_value("InvoiceId")
    invoice_date = get_field_value("InvoiceDate")
    due_date = get_field_value("DueDate")
    invoice_total = get_field_value("InvoiceTotal")
    subtotal = get_field_value("SubTotal")
    total_tax = get_field_value("TotalTax")

    print(f"Fornecedor: {vendor_name}")
    print(f"Cliente: {customer_name}")
    print(f"ID da Fatura: {invoice_id}")
    print(f"Data da Fatura: {invoice_date}")
    print(f"Data de Vencimento: {due_date}")
    print(f"Subtotal: {subtotal}")
    print(f"Imposto: {total_tax}")
    print(f"Total: {invoice_total}")

    print("\n\n--- RELATÓRIO DE VERIFICAÇÃO ANTI-FRAUDE ---")

    if not all([vendor_name, invoice_id, invoice_date, invoice_total]):
        relatorio_fraude.append("ALERTA: Informações críticas (Fornecedor, ID, Data ou Total) estão faltando.")

    if invoice_date and due_date:
        if invoice_date > due_date:
            relatorio_fraude.append(f"ALERTA: A data da fatura ({invoice_date}) é posterior à data de vencimento ({due_date}).")

    if invoice_date and isinstance(invoice_date, datetime.date):
        if invoice_date > datetime.now().date():
            relatorio_fraude.append(f"ALERTA: A data da fatura ({invoice_date}) está no futuro.")

    if subtotal and total_tax and invoice_total:
        try:
            calculo_total = round(subtotal.amount + total_tax.amount, 2)
            if abs(calculo_total - invoice_total.amount) > 0.01:
                relatorio_fraude.append(f"ALERTA: A soma do subtotal ({subtotal.amount}) e imposto ({total_tax.amount}) é {calculo_total}, o que não corresponde ao total da fatura ({invoice_total.amount}).")
        except (TypeError, AttributeError):
             relatorio_fraude.append("AVISO: Não foi possível realizar a validação matemática dos totais.")

    if not relatorio_fraude:
        print("✅ Nenhuma inconsistência básica encontrada.")
    else:
        for item in relatorio_fraude:
            print(f"- {item}")
else:
    print("Nenhum documento foi processado ou encontrado no resultado.")